<a href="https://www.kaggle.com/code/nilotpaldhar/finger-print-based-blood-group?scriptVersionId=319598092" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports & Device

In [ ]:
import os, glob, cv2, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Dataset

In [ ]:
def find_dataset_root(base="/kaggle/input", extensions=("*.BMP","*.bmp","*.png","*.jpg","*.jpeg")):
    """Walk /kaggle/input and find the folder that contains images."""
    print(f"Scanning {base} for image files...")
    for ext in extensions:
        found = glob.glob(os.path.join(base, "**", ext), recursive=True)
        if found:
            print(f"Found {len(found)} images with extension '{ext}'")
            return found
    return []

all_images = find_dataset_root()

if not all_images:
    print("\n NO IMAGES FOUND. Run the cell below to check your input folder:")
    print("   !ls -R /kaggle/input/")
else:
    data = []
    for fp in all_images:
        blood_group = os.path.basename(os.path.dirname(fp))
        image_id    = os.path.splitext(os.path.basename(fp))[0]
        data.append({"image_id": image_id, "blood_group": blood_group, "path": fp})

    df = pd.DataFrame(data)

    # Remove rows where folder name is not a valid blood group
    valid_groups = {"A+","A-","B+","B-","AB+","AB-","O+","O-"}
    df = df[df["blood_group"].isin(valid_groups)].reset_index(drop=True)

    print(f"\n Total valid images loaded: {len(df)}")
    print("\nBlood Group Distribution:")
    print(df["blood_group"].value_counts())

# Label Encoding

In [ ]:
df.head()

In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["blood_group"])
num_classes = len(le.classes_)

print("Blood Group → Label Mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")
print(f"\nTotal classes: {num_classes}")

# EDA Dashboard

In [ ]:
sns.set(style="whitegrid")
df_eda      = df.dropna(subset=["blood_group"]).copy()
class_counts = df_eda["blood_group"].value_counts().sort_index()
COLORS       = plt.cm.Paired(np.linspace(0, 1, len(class_counts)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes      = axes.flatten()

bars = axes[0].bar(class_counts.index, class_counts.values, color=COLORS)
axes[0].bar_label(bars, fmt="%d", padding=3)
axes[0].set_title("1. Blood Group Distribution", fontweight="bold")
axes[0].set_ylabel("Number of Images")

axes[1].pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%",
            startangle=140, colors=COLORS, explode=[0.03]*len(class_counts))
axes[1].set_title("2. Percentage Breakdown", fontweight="bold")

df_eda["rh_factor"] = df_eda["blood_group"].str.extract(r"([+-])")
rh_counts = df_eda["rh_factor"].value_counts()
rh_bars   = axes[2].bar(rh_counts.index, rh_counts.values, color=["#E74C3C","#3498DB"])
axes[2].bar_label(rh_bars, fmt="%d", padding=3)
axes[2].set_title("3. Rh Factor Distribution", fontweight="bold")

df_eda["abo_group"] = df_eda["blood_group"].str.replace(r"[+-]","",regex=True)
abo_counts = df_eda["abo_group"].value_counts()
abo_bars   = axes[3].bar(abo_counts.index, abo_counts.values,
                          color=plt.cm.Set2(np.linspace(0,1,4)))
axes[3].bar_label(abo_bars, fmt="%d", padding=3)
axes[3].set_title("4. ABO Group Distribution", fontweight="bold")

max_count       = class_counts.max()
imbalance_ratio = (class_counts / max_count * 100).sort_values()
axes[4].barh(imbalance_ratio.index, imbalance_ratio.values,
             color=plt.cm.RdYlGn(imbalance_ratio.values / 100))
axes[4].axvline(x=100, color="red", linestyle="--", alpha=0.5)
axes[4].set_title("5. Imbalance Ratio (% of Majority)", fontweight="bold")

axes[5].pie([80, 20], labels=["Train","Val"],
            autopct="%1.1f%%", colors=["#2980B9","#8E44AD"], startangle=90)
axes[5].set_title("6. Dataset Split Plan (80/20)", fontweight="bold")

plt.suptitle(
    f"Fingerprint-Based Blood Group — EDA Dashboard\n"
    f"Total: {len(df_eda):,} images | {len(class_counts)} Classes",
    fontsize=18, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("fingerprint_eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA saved as fingerprint_eda_dashboard.png")

# Sample Visualisation

In [ ]:
if len(df) >= 12:
    plt.figure(figsize=(15, 10))
    samples = df.sample(12, random_state=42)
    for i, (_, row) in enumerate(samples.iterrows()):
        img = cv2.imread(row["path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(3, 4, i+1)
        plt.imshow(img)
        plt.title(f"Group: {row['blood_group']}", fontsize=11, fontweight="bold")
        plt.axis("off")
    plt.suptitle("Fingerprint Samples by Blood Group", fontsize=15)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    print("Not enough images to sample 12.")

# Dataset Class  

In [ ]:
class FingerprintDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img   = cv2.imread(self.df.iloc[idx]["path"])
        img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label = self.df.iloc[idx]["label"]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((448, 448)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.RandomVerticalFlip(p=0.4),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # slight shift
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomGrayscale(p=0.05),  
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Train / Val / Test Split   

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

train_loader = DataLoader(FingerprintDataset(train_df, train_transform),
                          batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(FingerprintDataset(val_df,   test_transform),
                          batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(FingerprintDataset(test_df,  test_transform),
                          batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}  Labels: {labels.shape}")

# Model 

In [ ]:
model = models.resnet50(weights="IMAGENET1K_V2")

for p in model.parameters():
    p.requires_grad = False

for p in model.layer3.parameters():
    p.requires_grad = True
for p in model.layer4.parameters():
    p.requires_grad = True

num_features = model.fc.in_features   # 2048
model.fc = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.BatchNorm1d(1024),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, num_classes),
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# Loss, Optimizer, Scheduler    

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 

optimizer = torch.optim.AdamW([
    {"params": model.layer3.parameters(), "lr": 1e-5},  
    {"params": model.layer4.parameters(), "lr": 1e-5},  
    {"params": model.fc.parameters(),     "lr": 1e-4},   
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-7
)


# Training Loop    

In [ ]:
from tqdm import tqdm

history = {"train_loss": [], "train_acc": [], "val_acc": [], "test_acc": []}

def evaluate(loader, desc="Eval"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc=desc, leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds      = model(imgs).argmax(dim=1)
            correct   += (preds == lbls).sum().item()
            total     += lbls.size(0)
    return (correct / total) * 100


def train(epochs=100, patience=15):
    best_val_acc      = 0.0
    epochs_no_improve = 0

    for epoch in range(epochs):
        
        model.train()
        train_loss = train_correct = train_total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for imgs, lbls in pbar:
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss    += loss.item()
            train_correct += (out.argmax(1) == lbls).sum().item()
            train_total   += lbls.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss  = train_loss / len(train_loader)
        train_acc = (train_correct / train_total) * 100

        val_acc  = evaluate(val_loader,  desc=f"Epoch {epoch+1} [Val]")
        test_acc = evaluate(test_loader, desc=f"Epoch {epoch+1} [Test]")

        scheduler.step()

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["test_acc"].append(test_acc)

        current_lr = optimizer.param_groups[-1]["lr"]
        print(f"\nEpoch {epoch+1:03d} | Loss: {avg_loss:.4f} | "
              f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}% | "
              f"Test: {test_acc:.2f}% | LR: {current_lr:.2e}")

        if val_acc > best_val_acc:
            best_val_acc      = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), "blood_group_resnet50_best.pth")
            print(f" ⭐ New Best! Val: {val_acc:.2f}% | Test: {test_acc:.2f}%")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{patience})")

        if epochs_no_improve >= patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}. "
                  f"Best Val Acc: {best_val_acc:.2f}%")
            break

    print(f"\n Training complete. Best Val Accuracy: {best_val_acc:.2f}%")


train(epochs=30, patience=10)

# Training Graphs    

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("FingerPrint2BloodGroup — Training Summary",
             fontsize=16, fontweight="bold")

axes[0].plot(ep, history["train_acc"], marker="s", label="Train",
             color="#ed8936", linewidth=2, linestyle="--")
axes[0].plot(ep, history["val_acc"],   marker="o", label="Val",
             color="#4299e1", linewidth=2)
axes[0].plot(ep, history["test_acc"],  marker="^", label="Test",
             color="#68d391", linewidth=2, linestyle=":")
axes[0].set_title("Accuracy (Train / Val / Test)", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy (%)")
axes[0].legend(loc="lower right"); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history["train_loss"], marker="o",
             color="#fc8181", linewidth=2, label="Train Loss")
axes[1].set_title("Training Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("fingerprint_training_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph saved as fingerprint_training_summary.png")

# Save Model & Classes

In [ ]:
model.load_state_dict(torch.load("blood_group_resnet50_best.pth", map_location=device))

torch.save(model.state_dict(), "blood_group_resnet50_best.pth")
np.save("blood_group_classes.npy", le.classes_)
print(" Model and label classes saved successfully.")
print(f"   Classes: {le.classes_}")

# Confusion Matrix & Classification Report

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        _, predicted = torch.max(out, 1)
        y_true.extend(lbls.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted Blood Group", fontweight="bold")
plt.ylabel("True Blood Group",      fontweight="bold")
plt.title("Confusion Matrix: Fingerprint Blood Group Prediction", fontsize=14)
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Inference Visualisation    

In [ ]:
import random

def visualize_random_predictions(n=6):
    model.eval()
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    random_indices = random.sample(range(len(val_df)), n)
    
    plt.figure(figsize=(15, 10))
    
    for i, idx in enumerate(random_indices):
        img_path = val_df.iloc[idx]['path']
        true_label_idx = val_df.iloc[idx]['label']
        true_cls = le.inverse_transform([true_label_idx])[0]
        
        raw_img = cv2.imread(img_path)
        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        
        input_tensor = test_transform(raw_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = torch.max(output, 1)[1].item()
            pred_cls = le.inverse_transform([pred_idx])[0]
        
        display_img = input_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)
        display_img = np.clip(std * display_img + mean, 0, 1)
        
        color = "green" if true_cls == pred_cls else "red"
        plt.subplot(2, 3, i+1)
        plt.imshow(display_img)
        plt.title(f"True: {true_cls}\nPred: {pred_cls}", 
                  color=color, fontsize=12, fontweight="bold")
        plt.axis("off")

    plt.suptitle("Random Inference Check — Fingerprint Blood Group Prediction", fontsize=16)
    plt.tight_layout()
    plt.savefig("random_inference_check.png", dpi=150, bbox_inches="tight")
    plt.show()

visualize_random_predictions(n=6)